# 개별종목 조합E — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합E 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합E의 피처 값만 지정합니다.
import json

COMBINATION = 'E'
FEATURE_COLUMNS = (
    'sector_ret_5',
    'sector_ret_20',
    'relative_ret_5_sector',
    'relative_ret_20_sector',
    'relative_ret_5_market',
    'sector_hv_20',
    'sector_beta_60',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합E 피처: ('sector_ret_5', 'sector_ret_20', 'relative_ret_5_sector', 'relative_ret_20_sector', 'relative_ret_5_market', 'sector_hv_20', 'sector_beta_60')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4929,0.5012,-0.0083,0.3494,0.1537,0.2632
1,2,balanced,980,20150123,20150421,0.3612,0.3978,-0.0366,0.3000,0.1609,0.2436
2,3,balanced,1210,20151228,20160328,0.3686,0.3762,-0.0075,0.3620,0.2804,0.3318
3,4,balanced,1439,20161202,20170228,0.4350,0.4617,-0.0267,0.3493,0.1920,0.2893
4,5,balanced,1669,20171113,20180207,0.4015,0.3901,0.0114,0.3817,0.3568,0.3791
5,6,balanced,1899,20181024,20190118,0.3934,0.3725,0.0209,0.3917,0.3630,0.3822
6,7,balanced,2129,20190930,20191224,0.4467,0.4781,-0.0314,0.3589,0.2514,0.3333
7,8,balanced,2359,20200902,20201130,0.4037,0.3476,0.0561,0.3996,0.3824,0.3950
8,9,balanced,2589,20210806,20211105,0.3687,0.3916,-0.0230,0.3615,0.2735,0.3284
9,10,balanced,2818,20220714,20221012,0.3440,0.3454,-0.0014,0.3426,0.2401,0.3003


,OOS 폴드 평균
accuracy,0.4002
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0033
macro_f1,0.3639
down_recall,0.2706
core_harmonic_mean,0.3293


재실행 명령: python scripts/run_stock_model_experiment.py
